## 1.API Configuration

In [11]:
import os
from getpass import getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"]=getpass("Enter your OpenAI API Key:")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

GENERATION_MODEL = "gpt-4.1-mini"

print("OpenAI Client configured successfully.")
print("Model:",GENERATION_MODEL)

OpenAI Client configured successfully.
Model: gpt-4.1-mini


## 2.Define 5 functions or tools in python format with return

Note : Capture & raise expectations if you think will raise later.

In [31]:
def employee_lookup(employee_id):
    employees={101:{"name":"Sujin","department":"AI Engineering Team"},
               102:{"name":"Divo","department":"Operations Team"},
               103:{"name":"Ani","department":"AI Research Team"}}

    return employees.get(employee_id,{"error":"Employee Not Found."})

def calculate_leave_balance(total_leaves,leaves_taken):
    return {"Remaining leaves:",total_leaves-leaves_taken}

def calculate(expression):
    allowed=set("0123456789+-*(). %")

    if not expression or any(char not in allowed for char in expression):
        return {"error":"Only basic arithmetic operations are allowed."}
    try:
        return {"result": eval(expression, {"__buildins__":{}},{})}
    except Exception:
        return {"error": "Invalid Arithmetic operation"}

def send_email(to,subject,body):
    return {"status":"simulated","message":f"Email prepared for {to} with subject '{subject}'."}

def book_meeting(title,attendee,time):
    return {"status":"simulated","message":f"Meeting '{title}' prepared with {attendee} at {time}."}

## 2.2 Now define in OpenAI SDK format

In [32]:
TOOLS =[
    {
        "type":"function",
        "name":"employee_lookup",
        "description":"Look up and Employee's name and Department using employee ID.",
        "parameters":{
            "type":"object",
            "properties":{
                "employee_id":{"type":"integer"}
            },
            "required":["employee_id"],
            "additionalProperties":False
        }
    },
    {
        "type":"function",
        "name":"calculate_leave_balance",
        "description":"Calculate remaining leave from total ans annual leaves.",
        "parameters":{
            "type":"object",
            "properties":{
                "total_leaves":{"type":"integer"},
               "leaves_taken":{"type":"integer"}
            },
            "required":["total_leaves","leaves_taken"],
            "additionalProperties":False
        }
    },
    {
        "type":"function",
        "name":"calculate",
        "description":"Performa a basic arithmetic calculation.",
        "parameters":{
            "type":"object",
            "properties":{
                "expression":{"type":"string"}
                },
            "required":["expression"],
            "additionalProperties":False
    }
    },
    {
        "type":"function",
        "name":"send_email",
        "description":"Prepare a Simulated email.",
        "parameters":{
            "type":"object",
            "properties":{
                "to":{"type":"string"},
                "subject":{"type":"string"},
                "body":{"type":"string"}
            },
            "required":["to","subject","body"],
            "additionalProperties":False
        }
        
    },
    {
        "type":"function",
        "name":"book_meeting",
        "description":"Prepare a simluated meeting booking.",
        "parameters":{
            "type":"object",
            "properties":{
                "title":{"type":"string"},
                "attendee":{"type":"string"},
                "time":{"type":"string"}
            },
            "required":["title","attendee","time"],
            "additionalProperties":False
        }
    }
]

In [8]:
print(TOOLS)

[{'type': 'function', 'name': 'employee_lookup', 'description': "Look up and Employee's name and Department using employee ID.", 'parameters': {'type': 'object', 'properties': {'employee_id': {'type': 'integer'}}, 'required': ['employee_id'], 'additionalProperties': False}}, {'type': 'function', 'name': 'calculate_leave_balance', 'description': 'Calculate remaining leave from total ans annual leaves.', 'parameters': {'type': 'object', 'properties': {'total_leaves': {'type': 'integer'}, 'leaves_taken': {'type': 'integer'}}, 'required': ['total_leaves', 'leaves_taken'], 'additionalProperties': False}}, {'type': 'function', 'name': 'calculate', 'description': 'Performa a basic arithmetic calculation.', 'parameters': {'type': 'object', 'properties': {'expression': {'type': 'string'}}, 'required': ['expression'], 'additionalProperties': False}}, {'type': 'function', 'name': 'send_email', 'description': 'Prepare a Simulated email.', 'parameters': {'type': 'object', 'properties': {'to': {'typ

## 3.Send the API request to get the tool call

In [33]:
question ="Who is Employee 102?"

response =client.responses.create(model=GENERATION_MODEL,input=question,tools=TOOLS)

print(response.output)

[ResponseFunctionToolCall(arguments='{"employee_id":102}', call_id='call_7FvSceCB32KQKcjVGJ3lZzLR', name='employee_lookup', type='function_call', id='fc_0c89c444da3dd747006aae48ac12a487d19afe40641f7d0201', async_=None, caller=None, namespace=None, status='completed')]


In [34]:
for item in response.output:
    if item.type == "function_call":
        print("Selected Tool:",item.name)
        print("Arguments:",item.arguments)

Selected Tool: employee_lookup
Arguments: {"employee_id":102}


## What if teh input is now....

Find employee 103. Then prepare a simluated email to HR that mentions the employee's name and department and asks HR to confirm the employee's leave balance.

## ==========================================================================

## Prerequisite : When dealing with multiple tools, we need to store  the Tools Registry

In [35]:
TOOL_REGISTRY ={"employee_lookup":employee_lookup,
               "calculate_leave_balance":calculate_leave_balance,
               "calculate":calculate,
                "send_email":send_email,
                "book_meeting":book_meeting}

# Interview Question

## How to Create an AI Agent without using any Frameworks?

### 10 steps to create andrun the Agent......

1. Receive the user input
 
2.Store the conversation

3.Start Agent Loop

4.Send request to LLM

5.Check for Tool call

6.Save LLM response

7.Execute selected tool

8.Get tool result

9.Send tool result back to LLM

10.Repeat until the final answer or Maximum steps reached

### Script:

In [69]:
import json

def run_agent(user_input,max_steps=5):
    input_items=[{"role":"user","content":user_input}]

    for step in range(1,max_steps+1):

        response = client.responses.create(model=GENERATION_MODEL,
                                          input=input_items,
                                          tools=TOOLS)

        tool_calls=[item for item in response.output if item.type=="function_call"]

        if not tool_calls:
            return response.output_text

        input_items.extend([item.model_dump(by_alias=True) for item in response.output])   #--Part 1 is done

        for tool_call in tool_calls:
            if tool_call.name not in TOOL_REGISTRY:
                raise ValueError(f"Unknown tool requested: {tool_call.name}")

            arguments = json.loads(tool_call.arguments)

            result = TOOL_REGISTRY[tool_call.name](**arguments)

            print(f"step {step}")
            print("Tool:",tool_call.name)
            print("Arguments",arguments)
            print("Result",result)

            input_items.append({
                    "type":"function_call_output",
                    "call_id":tool_call.call_id,
                    "output":json.dumps(result)
                })
    return "Maximum agent steps reached."

In [70]:
run_agent("Find employee 102 and tell me their name and department.")

step 1
Tool: employee_lookup
Arguments {'employee_id': 102}
Result {'name': 'Divo', 'department': 'Operations Team'}


'Employee 102 is named Divo, and they belong to the Operations Team department.'

In [71]:
run_agent("Find employee 103. Then prepare a simulated email to HR that mentions the employee's name and department and asks HR to confirm the employee's leave balance.")

step 1
Tool: employee_lookup
Arguments {'employee_id': 103}
Result {'name': 'Ani', 'department': 'AI Research Team'}
step 2
Tool: send_email
Arguments {'to': 'HR@example.com', 'subject': 'Leave Balance Confirmation Request for Employee Ani', 'body': 'Dear HR Team,\n\nI hope this message finds you well. Could you please confirm the leave balance for our employee Ani from the AI Research Team?\n\nThank you for your assistance.\n\nBest regards,\n[Your Name]'}
Result {'status': 'simulated', 'message': "Email prepared for HR@example.com with subject 'Leave Balance Confirmation Request for Employee Ani'."}


"I found employee 103, Ani, who is in the AI Research Team. I have prepared a simulated email to HR asking them to confirm Ani's leave balance. Would you like me to assist with anything else?"